In [1]:
import anndata as ad
import pandas as pd
import pickle

In [2]:
import numpy as np

In [3]:
ratio = 0.25

In [4]:
op3_train_subsample = ad.read_h5ad('./data_mol_emb/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('./data_mol_emb/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad')

In [5]:
df = pd.read_csv('./data_mol_emb/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv')

In [6]:
path = './data_mol_emb/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl'
with open(path, 'rb') as fp:
    op3_emb = pickle.load(fp)

In [7]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [8]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [9]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [10]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [11]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [12]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad('./data/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')

In [13]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad('./data/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [14]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('./data/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)